## 前処理04

本ファイルは以下の処理を行うコードである。

1. `/workspace/datasets/statcast-customized/preprocess_03`データセットを読みだす。

2. `description`カラムを`swing_attempt`と`swing_result`に成形し、`/workspace/datasets/statcast-customized/preprocess_04`に保存する。

In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
load_root_path = '/workspace/datasets/statcast-customized-tmp/preprocess_03/'
save_root_path = '/workspace/datasets/statcast-customized-tmp/preprocess_04/'

In [3]:
def get_parquet_file_path_list(load_root_path):
    file_path_list = []
    # Walk through the directory and its subdirectories to find all Parquet files
    for root, dirs, files in os.walk(load_root_path):
        for file in files:
            if file.endswith('.parquet'):
                file_path_list.append(os.path.join(root, file))
    # Sort the file paths to ensure consistent order
    file_path_list.sort()
    return file_path_list

In [4]:
def load_parquet(path: str) -> pd.DataFrame:
    """Read a Parquet file and return a pandas DataFrame."""
    return pd.read_parquet(path)

In [5]:
parquet_file_path_list = get_parquet_file_path_list(load_root_path)
for file_path in parquet_file_path_list:
    print(f'Load: {file_path}')
print(f'Number of Parquet files: {len(parquet_file_path_list)}')

Load: /workspace/datasets/statcast-customized-tmp/preprocess_03/statcast_2017_04.parquet
Load: /workspace/datasets/statcast-customized-tmp/preprocess_03/statcast_2017_05.parquet
Load: /workspace/datasets/statcast-customized-tmp/preprocess_03/statcast_2017_06.parquet
Load: /workspace/datasets/statcast-customized-tmp/preprocess_03/statcast_2017_07.parquet
Load: /workspace/datasets/statcast-customized-tmp/preprocess_03/statcast_2017_08.parquet
Load: /workspace/datasets/statcast-customized-tmp/preprocess_03/statcast_2017_09.parquet
Load: /workspace/datasets/statcast-customized-tmp/preprocess_03/statcast_2017_10.parquet
Load: /workspace/datasets/statcast-customized-tmp/preprocess_03/statcast_2017_11.parquet
Load: /workspace/datasets/statcast-customized-tmp/preprocess_03/statcast_2018_03.parquet
Load: /workspace/datasets/statcast-customized-tmp/preprocess_03/statcast_2018_04.parquet
Load: /workspace/datasets/statcast-customized-tmp/preprocess_03/statcast_2018_05.parquet
Load: /workspace/data

In [6]:
# データの確認
df_one_month = load_parquet(parquet_file_path_list[0])
with pd.option_context('display.max_columns', None, 'display.max_rows', None):
    display(df_one_month.head(20))

,at_bat_id,description,bb_type,launch_speed,launch_angle,hit_distance_sc,p_throws,pitch_type,release_speed,release_spin_rate,pfx_x,pfx_z,plate_x,plate_z,batter,stand,base_out_state,count_state,inning_clipped,is_inning_top,diff_score_clipped,pitch_number_clipped
0,0,ball,None,NaN,NaN,NaN,R,FF,94.9,2146.0,-0.78,1.10,0.281634,4.334447,656941,L,0,0,1,True,0,1
1,0,foul,None,NaN,NaN,NaN,R,SI,95.9,2144.0,-1.59,0.84,-0.578784,2.324124,656941,L,0,3,1,True,0,2
2,0,ball,None,NaN,NaN,NaN,R,SI,97.3,2015.0,-1.34,0.55,-0.014284,1.326134,656941,L,0,4,1,True,0,3
3,0,hit_into_play,line_drive,96.1,18.0,256.0,R,SL,84.6,2194.0,1.10,-0.17,0.135816,1.943402,656941,L,0,7,1,True,0,4
4,1,called_strike,None,NaN,NaN,NaN,R,SI,98.4,2066.0,-1.28,0.68,0.421448,1.833604,592178,R,1,0,1,True,0,1
5,1,ball,None,NaN,NaN,NaN,R,SI,96.3,2086.0,-1.20,0.68,0.624980,2.677019,592178,R,1,1,1,True,0,2
6,1,ball,None,NaN,NaN,NaN,R,SL,86.6,2333.0,1.42,0.65,2.357224,1.320178,592178,R,1,4,1,True,0,3
7,1,called_strike,None,NaN,NaN,NaN,R,SL,86.1,2261.0,1.08,-0.03,0.263640,1.705923,592178,R,1,7,1,True,0,4
8,1,swinging_strike,None,NaN,NaN,NaN,R,FF,99.9,2278.0,-0.85,1.19,-0.540721,3.304990,592178,R,1,8,1,True,0,5
9,2,ball,None,NaN,NaN,NaN,R,FF,99.3,2144.0,-0.92,1.21,-0.226224,3.970468,519203,L,9,0,1,True,0,1


In [7]:
def get_unique_values_and_counts_from_column(column_name: str, parquet_files: list) -> dict:
    """Get unique values and their counts from a specified column in a DataFrame."""
    value_counts = {}
    for file_path in parquet_files:
        df = load_parquet(file_path)
        counts = df[column_name].value_counts()
        for value, count in counts.items():
            if value in value_counts:
                value_counts[value] += count
            else:
                value_counts[value] = count
    value_counts = dict(sorted(value_counts.items(), key=lambda item: item[1], reverse=True))
    return value_counts

In [8]:
unique_descriptions = get_unique_values_and_counts_from_column("description", parquet_file_path_list)
df_unique_descriptions = pd.DataFrame(list(unique_descriptions.items()), columns=["description", "count"])
display(df_unique_descriptions.head(20))
print(f'Number of unique descriptions: {len(unique_descriptions)}')

,description,count
0,ball,1935354
1,foul,1018256
2,hit_into_play,1016206
3,called_strike,940593
4,swinging_strike,597528
5,blocked_ball,132394
6,foul_tip,53779
7,swinging_strike_blocked,38143
8,automatic_ball,21213
9,hit_by_pitch,16491


Number of unique descriptions: 16


In [9]:
conversion_table = {
    "hit_into_play": {"swing_attempt": True, "swing_result": "hit_into_play"},
    "foul": {"swing_attempt": True, "swing_result": "foul"},
    "foul_tip": {"swing_attempt": True, "swing_result": "foul_tip"},
    "foul_bunt": {"swing_attempt": True, "swing_result": "foul_bunt"},
    "foul_pitchout": {"swing_attempt": True, "swing_result": "foul_pitchout"},
    "swinging_strike": {"swing_attempt": True, "swing_result": "swinging_strike"},
    "swinging_strike_blocked": {"swing_attempt": True, "swing_result": "swinging_strike_blocked"},
    "missed_bunt": {"swing_attempt": True, "swing_result": "missed_bunt"},
    "bunt_foul_tip": {"swing_attempt": True, "swing_result": "bunt_foul_tip"},
    "ball": {"swing_attempt": False, "swing_result": np.nan},
    "blocked_ball": {"swing_attempt": False, "swing_result": np.nan},
    "automatic_ball": {"swing_attempt": False, "swing_result": np.nan},
    "called_strike": {"swing_attempt": False, "swing_result": np.nan},
    "automatic_strike": {"swing_attempt": False, "swing_result": np.nan},
    "hit_by_pitch": {"swing_attempt": False, "swing_result": np.nan},
    "pitchout": {"swing_attempt": False, "swing_result": np.nan},
}

In [10]:
# unique_descriptions が dict の場合は DataFrame に変換
if isinstance(unique_descriptions, dict):
    unique_descriptions = pd.DataFrame(
        list(unique_descriptions.items()),
        columns=["description", "count"]
    )
elif isinstance(unique_descriptions, pd.DataFrame):
    unique_descriptions = unique_descriptions.copy()
else:
    raise TypeError("unique_descriptions は dict または pandas.DataFrame である必要があります。")

# conversion_table を適用して新しい列を追加
unique_descriptions[["swing_attempt", "swing_result"]] = unique_descriptions["description"].apply(
    lambda x: pd.Series(
        conversion_table.get(x, {"swing_attempt": np.nan, "swing_result": np.nan})
    )
)

# 必要な列の順序を指定
cols = ["swing_attempt", "swing_result", "description", "count"]
unique_descriptions = unique_descriptions[cols]

display(unique_descriptions.head(20))

,swing_attempt,swing_result,description,count
0,False,NaN,ball,1935354
1,True,foul,foul,1018256
2,True,hit_into_play,hit_into_play,1016206
3,False,NaN,called_strike,940593
4,True,swinging_strike,swinging_strike,597528
5,False,NaN,blocked_ball,132394
6,True,foul_tip,foul_tip,53779
7,True,swinging_strike_blocked,swinging_strike_blocked,38143
8,False,NaN,automatic_ball,21213
9,False,NaN,hit_by_pitch,16491


In [13]:
def add_swing_columns_from_description(df: pd.DataFrame, conversion_table: dict) -> pd.DataFrame:
    """
    descriptionカラムを conversion_table で変換し、
    swing_attempt / swing_result カラムを追加したDataFrameを返す。
    """
    if "description" not in df.columns:
        raise KeyError("入力DataFrameに 'description' カラムがありません。")

    mapped = df["description"].map(conversion_table)

    result = df.copy()
    result["swing_attempt"] = mapped.map(
        lambda x: x.get("swing_attempt", np.nan) if isinstance(x, dict) else np.nan
    )
    result["swing_result"] = mapped.map(
        lambda x: x.get("swing_result", np.nan) if isinstance(x, dict) else np.nan
    )

    # description カラムを削除
    result = result.drop(columns=["description"])
    # 列の順序を at_bat_id, swing_attempt, swing_result, その他の列 の順番にする
    other_cols = [col for col in result.columns if col not in ["at_bat_id", "swing_attempt", "swing_result"]]
    result = result[["at_bat_id", "swing_attempt", "swing_result"] + other_cols]

    return result

In [14]:
df_one_month_with_swing = add_swing_columns_from_description(df_one_month, conversion_table)
with pd.option_context('display.max_columns', None, 'display.max_rows', None):
    display(df_one_month_with_swing.head(10))

,at_bat_id,swing_attempt,swing_result,bb_type,launch_speed,launch_angle,hit_distance_sc,p_throws,pitch_type,release_speed,release_spin_rate,pfx_x,pfx_z,plate_x,plate_z,batter,stand,base_out_state,count_state,inning_clipped,is_inning_top,diff_score_clipped,pitch_number_clipped
0,0,False,NaN,None,NaN,NaN,NaN,R,FF,94.9,2146.0,-0.78,1.10,0.281634,4.334447,656941,L,0,0,1,True,0,1
1,0,True,foul,None,NaN,NaN,NaN,R,SI,95.9,2144.0,-1.59,0.84,-0.578784,2.324124,656941,L,0,3,1,True,0,2
2,0,False,NaN,None,NaN,NaN,NaN,R,SI,97.3,2015.0,-1.34,0.55,-0.014284,1.326134,656941,L,0,4,1,True,0,3
3,0,True,hit_into_play,line_drive,96.1,18.0,256.0,R,SL,84.6,2194.0,1.10,-0.17,0.135816,1.943402,656941,L,0,7,1,True,0,4
4,1,False,NaN,None,NaN,NaN,NaN,R,SI,98.4,2066.0,-1.28,0.68,0.421448,1.833604,592178,R,1,0,1,True,0,1
5,1,False,NaN,None,NaN,NaN,NaN,R,SI,96.3,2086.0,-1.20,0.68,0.624980,2.677019,592178,R,1,1,1,True,0,2
6,1,False,NaN,None,NaN,NaN,NaN,R,SL,86.6,2333.0,1.42,0.65,2.357224,1.320178,592178,R,1,4,1,True,0,3
7,1,False,NaN,None,NaN,NaN,NaN,R,SL,86.1,2261.0,1.08,-0.03,0.263640,1.705923,592178,R,1,7,1,True,0,4
8,1,True,swinging_strike,None,NaN,NaN,NaN,R,FF,99.9,2278.0,-0.85,1.19,-0.540721,3.304990,592178,R,1,8,1,True,0,5
9,2,False,NaN,None,NaN,NaN,NaN,R,FF,99.3,2144.0,-0.92,1.21,-0.226224,3.970468,519203,L,9,0,1,True,0,1


In [15]:
def get_features_and_save_parquet(parquet_file_path, save_root_path):
    df = load_parquet(parquet_file_path)
    df_with_features = add_swing_columns_from_description(df, conversion_table)

    # Create the save path by replacing the load root with the save root
    relative_path = os.path.relpath(parquet_file_path, load_root_path)
    save_path = os.path.join(save_root_path, relative_path)

    # Ensure the directory exists
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    # Save the extracted DataFrame to a new Parquet file
    df_with_features.to_parquet(save_path, index=False)
    print(f'Saved extracted data to: {save_path}')

In [16]:
for parquet_file_path in parquet_file_path_list:
    get_features_and_save_parquet(parquet_file_path, save_root_path)

Saved extracted data to: /workspace/datasets/statcast-customized-tmp/preprocess_04/statcast_2017_04.parquet
Saved extracted data to: /workspace/datasets/statcast-customized-tmp/preprocess_04/statcast_2017_05.parquet
Saved extracted data to: /workspace/datasets/statcast-customized-tmp/preprocess_04/statcast_2017_06.parquet
Saved extracted data to: /workspace/datasets/statcast-customized-tmp/preprocess_04/statcast_2017_07.parquet
Saved extracted data to: /workspace/datasets/statcast-customized-tmp/preprocess_04/statcast_2017_08.parquet
Saved extracted data to: /workspace/datasets/statcast-customized-tmp/preprocess_04/statcast_2017_09.parquet
Saved extracted data to: /workspace/datasets/statcast-customized-tmp/preprocess_04/statcast_2017_10.parquet
Saved extracted data to: /workspace/datasets/statcast-customized-tmp/preprocess_04/statcast_2017_11.parquet
Saved extracted data to: /workspace/datasets/statcast-customized-tmp/preprocess_04/statcast_2018_03.parquet
Saved extracted data to: /wo